# How to predict at new locations

You have fitted a spatial model and want the outcome at units you did not
observe: new tracts, a region withheld for validation, or places where the
survey missed. `model.predict` returns the posterior predictive distribution
of those outcomes given every observed one. For each posterior draw it uses
the exact conditional Gaussian, the best predictor (BP) of
{cite:t}`goulard2017PredictionsSpatial`, so the result carries parameter
uncertainty instead of plugging in point estimates.

It also returns the *trend* predictor (TC), the model's mean at the new units
with no correction from observed neighbors. TC is the floor BP has to beat.
This page shows how to call `predict`, and how to check what the spatial
correction is worth on your data.

`predict` covers OLS, SLX, SAR, SDM, SEM and SDEM with Gaussian errors.
Models with `robust=True` raise, because the Gaussian conditional does not
hold for Student-t errors.

## Fit on the observed units

We simulate a SAR on a 20 × 20 rook grid, hold out four 4 × 4 blocks, and fit
on the rest. The training weights are the contiguity graph of the observed
units, row-standardized.

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from libpysal.graph import Graph

from neighbayes.dgp import simulate_sar
from neighbayes.models import SAR

GRID = 20


def held_out_blocks(n_side, rng, n_blocks=4, size=4):
    """Mask of ``n_blocks`` non-overlapping ``size`` x ``size`` blocks."""
    row, col = np.divmod(np.arange(n_side * n_side), n_side)
    slots = [
        (r, c)
        for r in range(0, n_side - size + 1, 5)
        for c in range(0, n_side - size + 1, 5)
    ]
    held = np.zeros(n_side * n_side, dtype=bool)
    for k in rng.choice(len(slots), n_blocks, replace=False):
        r0, c0 = slots[k]
        held |= (row >= r0) & (row < r0 + size) & (col >= c0) & (col < c0 + size)
    return held


gdf = simulate_sar(
    n_side=GRID,
    rho=0.6,
    beta=np.array([1.0, 2.0]),
    sigma=1.0,
    seed=0,
    create_gdf=True,
    geometry_type="polygon",
    contiguity="rook",
)
held = held_out_blocks(GRID, np.random.default_rng(0))
train = gdf[~held]
new = gdf[held]

model = SAR(
    formula="y ~ X_1",
    data=train,
    W=Graph.build_contiguity(train, rook=True).transform("r"),
)
model.fit(draws=500, tune=500, chains=2, random_seed=0, progressbar=False)
model.summary()

## Predict

`predict` needs the covariates of the new units and a weights matrix over
*all* units, observed and new. By default the new units come last in that
matrix. Here they sit among the observed ones, so `oos` gives their positions.
For a formula-built model, pass the new covariates as a DataFrame; the
formula's transforms are replayed on it.

In [ ]:
W_all = Graph.build_contiguity(gdf, rook=True).transform("r")
oos = np.flatnonzero(held)

pred = model.predict(new, W_all, oos=oos, random_seed=0)
pred

The `predictions` group holds three variables over `(chain, draw, obs_new)`:

- `y`, draws from the posterior predictive, for intervals;
- `y_bp`, the conditional mean at each draw, whose average over draws is the
  posterior predictive mean;
- `y_tc`, the trend at each draw, ignoring observed neighbors.

The coordinate `obs_new` carries the index of `new`.

In [ ]:
import arviz as az

p = pred.predictions
table = pd.DataFrame(
    {
        "observed": new["y"],
        "bp": p["y_bp"].mean(("chain", "draw")).values,
        "tc": p["y_tc"].mean(("chain", "draw")).values,
    },
    index=p["obs_new"].values,
)
hdi = az.hdi(p["y"], prob=0.9).transpose("obs_new", ...).values
table["lo90"], table["hi90"] = hdi[:, 0], hdi[:, 1]
table.head()

For models with lagged covariates (SLX, SDM, SDEM), `predict`
recomputes $WX$ for *every* unit on the full graph: an observed unit that
gains a new neighbor gets a new lagged covariate.

## Check what the correction is worth

BP improves on TC only through observed neighbors. A new unit deep inside a
held-out block has none, and its prediction falls back almost entirely to the
trend. {cite:t}`goulard2017PredictionsSpatial` found TC within 94–97% of BP's
efficiency on sparse weights, and closer still as more neighbors are missing.
So before relying on BP, compare the two on held-out data, stratified by how
many of each unit's neighbors were observed. We repeat the exercise over eight
simulated maps and report each stratum's gain with its Monte Carlo standard
error.

In [ ]:
A = (sp.csr_matrix(W_all.sparse) != 0).astype(int)
rows = []
for rep in range(8):
    rng = np.random.default_rng(rep)
    g = simulate_sar(
        n_side=GRID,
        rho=0.6,
        beta=np.array([1.0, 2.0]),
        sigma=1.0,
        seed=rep,
        create_gdf=True,
        geometry_type="polygon",
        contiguity="rook",
    )
    h = held_out_blocks(GRID, rng)
    tr = g[~h]
    m = SAR(
        formula="y ~ X_1",
        data=tr,
        W=Graph.build_contiguity(tr, rook=True).transform("r"),
    )
    m.fit(draws=500, tune=500, chains=2, random_seed=rep, progressbar=False)
    o = np.flatnonzero(h)
    pp = m.predict(g[h], W_all, oos=o, random_seed=rep).predictions
    y = g["y"].to_numpy()[o]
    rows.append(
        pd.DataFrame(
            {
                "observed_neighbors": np.asarray(A[o][:, ~h].sum(axis=1)).ravel(),
                "se_bp": (y - pp["y_bp"].mean(("chain", "draw")).values) ** 2,
                "se_tc": (y - pp["y_tc"].mean(("chain", "draw")).values) ** 2,
            }
        )
    )
errors = pd.concat(rows, ignore_index=True)
errors["gain"] = errors["se_tc"] - errors["se_bp"]
errors.groupby("observed_neighbors").agg(
    n=("gain", "size"),
    mse_bp=("se_bp", "mean"),
    mse_tc=("se_tc", "mean"),
    gain=("gain", "mean"),
    gain_se=("gain", "sem"),
).round(3)

Read `gain` against `gain_se`. For units with no observed neighbor the gain
is within two standard errors of zero; BP still draws on neighbors of
neighbors, but not enough to matter. The gain grows with each observed
neighbor. If your held-out units are mostly surrounded by other
missing units, the cheap trend predictor is as good as the correction; if
they are scattered among observed ones, the correction pays.

## When the conditional is not what you want

- `predict_in_sample()` gives each fitted unit's leave-one-out conditional,
  the in-sample BP of {cite:t}`goulard2017PredictionsSpatial`. It conditions
  on a posterior fitted to all the data, so it smooths rather than
  validates.
- To *score* held-out predictive accuracy across models, use
  [spatial block cross-validation](spatial_cv_demo.ipynb), which refits on
  each fold and evaluates this same conditional density.

The parameters come from the observed units' own weights, row-standardized
after the new units are removed, and the prediction uses the full weights:
the feasible practice of {cite:t}`goulard2017PredictionsSpatial`. When the
missing units are scattered, many observed units lose neighbors and the fit
drifts. On a 10 × 10 grid with 12% of units missing in a checkerboard, the
posterior mean of ρ fell from 0.57 to 0.42, and 90% intervals covered 95–96%
of held-out values instead of 90%. Compact missing regions cost far less.

## See also

- [How to run spatial block cross-validation](spatial_cv_demo.ipynb)
- [Supported Models](../models.md)